# MedTrack_DV – Milestone 1
## Notebook 02: Data Cleaning
**Purpose:** Clean all 4 datasets — remove duplicates, fix missing values, fix data types

## Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

RAW = '../data/raw/'

# Load all datasets
hmis_sheets      = pd.read_excel(RAW + 'Hospital_Management_System.xlsx', sheet_name=None)
df_appointments  = hmis_sheets['Appointment'].copy()
df_bed_records   = hmis_sheets['BedRecords'].copy()
df_department    = hmis_sheets['Department'].copy()
df_doctor        = hmis_sheets['Doctor'].copy()
df_nurse         = hmis_sheets['Nurse'].copy()
df_patients      = hmis_sheets['Patients'].copy()
df_ward          = hmis_sheets['Ward'].copy()
df_bed           = hmis_sheets['Bed'].copy()
df_medical       = hmis_sheets['MedicalRecord'].copy()
df_staff_shift   = hmis_sheets['StaffShift'].copy()
df_surgery       = hmis_sheets['SurgeryRecord'].copy()

df_beds_patients = pd.read_csv(RAW + 'beds_patients.csv')
df_beds_services = pd.read_csv(RAW + 'beds_services_weekly.csv')
df_beds_staff    = pd.read_csv(RAW + 'beds_staff.csv')
df_beds_schedule = pd.read_csv(RAW + 'beds_staff_schedule.csv')

df_readmission   = pd.read_csv(RAW + 'readmission_admission_data.csv')
df_mortality     = pd.read_csv(RAW + 'readmission_mortality_data.csv')

df_healthcare    = pd.read_csv(RAW + 'healthcare_dataset.csv')

print('All datasets loaded for cleaning!')

## Step 2: Data Profiling – Check Missing Values & Duplicates

In [ ]:
def profile_dataset(df, name):
    print(f'\n=== {name} ===')
    print(f'Shape: {df.shape}')
    print(f'Duplicates: {df.duplicated().sum()}')
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
    missing_df = missing_df[missing_df['Missing Count'] > 0]
    if len(missing_df) > 0:
        print('Missing Values:')
        display(missing_df)
    else:
        print('No missing values!')

profile_dataset(df_patients,    'HMIS - Patients')
profile_dataset(df_bed_records, 'HMIS - BedRecords (Admissions)')
profile_dataset(df_department,  'HMIS - Department')
profile_dataset(df_doctor,      'HMIS - Doctor')
profile_dataset(df_beds_patients,'Beds - Patients')
profile_dataset(df_beds_services,'Beds - Services Weekly')
profile_dataset(df_readmission, 'Readmission - Admission Data')
profile_dataset(df_healthcare,  'Healthcare Dataset')

## Step 3: Clean Dataset 1 – HMIS Tables

In [ ]:
# --- Clean Patients ---
df_patients.columns = [c.strip().lower() for c in df_patients.columns]
df_patients = df_patients.drop_duplicates(subset='patient_id')
df_patients['gender'] = df_patients['gender'].str.strip().str.upper()
df_patients['date_of_birth'] = pd.to_datetime(df_patients['date_of_birth'], errors='coerce')
df_patients['fname'] = df_patients['fname'].str.strip().str.title()
df_patients['lname'] = df_patients['lname'].str.strip().str.title()
df_patients['full_name'] = df_patients['fname'] + ' ' + df_patients['lname']
print(f'Patients cleaned: {df_patients.shape}')

In [ ]:
# --- Clean BedRecords (Admissions) ---
df_bed_records.columns = [c.strip().lower() for c in df_bed_records.columns]
df_bed_records = df_bed_records.drop_duplicates(subset='admission_id')
df_bed_records['admission_date'] = pd.to_datetime(df_bed_records['admission_date'], errors='coerce')
df_bed_records['discharge_date'] = pd.to_datetime(df_bed_records['discharge_date'], errors='coerce')
df_bed_records['length_of_stay_days'] = (df_bed_records['discharge_date'] - df_bed_records['admission_date']).dt.days
# Remove impossible LOS values
df_bed_records = df_bed_records[df_bed_records['length_of_stay_days'] >= 0]
df_bed_records['mode_of_payment'] = df_bed_records['mode_of_payment'].str.strip().str.title()
print(f'BedRecords cleaned: {df_bed_records.shape}')
display(df_bed_records.head(3))

In [ ]:
# --- Clean Department ---
df_department.columns = [c.strip().lower() for c in df_department.columns]
df_department = df_department.drop_duplicates(subset='dept_id')
df_department['dept_name'] = df_department['dept_name'].str.strip().str.title()
print(f'Department cleaned: {df_department.shape}')
display(df_department)

In [ ]:
# --- Clean Doctor ---
df_doctor.columns = [c.strip().lower() for c in df_doctor.columns]
df_doctor = df_doctor.drop_duplicates(subset='doct_id')
df_doctor['gender'] = df_doctor['gender'].str.strip().str.upper()
df_doctor['fname'] = df_doctor['fname'].str.strip().str.title()
df_doctor['lname'] = df_doctor['lname'].str.strip().str.title()
df_doctor['full_name'] = df_doctor['fname'] + ' ' + df_doctor['lname']
df_doctor['surgeon_type'] = df_doctor['surgeon_type'].str.strip().str.title()
print(f'Doctor cleaned: {df_doctor.shape}')

In [ ]:
# --- Clean Ward ---
df_ward.columns = [c.strip().lower() for c in df_ward.columns]
df_ward = df_ward.drop_duplicates(subset='ward_no')
df_ward['ward_name'] = df_ward['ward_name'].str.strip().str.title()
print(f'Ward cleaned: {df_ward.shape}')
display(df_ward.head())

## Step 4: Clean Dataset 2 – Beds Management

In [ ]:
# --- Clean Beds Patients ---
df_beds_patients.columns = [c.strip().lower().replace(' ', '_') for c in df_beds_patients.columns]
df_beds_patients = df_beds_patients.drop_duplicates(subset='patient_id')
df_beds_patients['arrival_date']    = pd.to_datetime(df_beds_patients['arrival_date'], errors='coerce')
df_beds_patients['departure_date']  = pd.to_datetime(df_beds_patients['departure_date'], errors='coerce')
df_beds_patients['length_of_stay']  = (df_beds_patients['departure_date'] - df_beds_patients['arrival_date']).dt.days
df_beds_patients['service']         = df_beds_patients['service'].str.strip().str.title()
df_beds_patients['name']            = df_beds_patients['name'].str.strip().str.title()
# Validate satisfaction scores
df_beds_patients['satisfaction'] = df_beds_patients['satisfaction'].clip(0, 10)
print(f'Beds Patients cleaned: {df_beds_patients.shape}')
display(df_beds_patients.head(3))

In [ ]:
# --- Clean Beds Services Weekly ---
df_beds_services.columns = [c.strip().lower().replace(' ', '_') for c in df_beds_services.columns]
df_beds_services['service'] = df_beds_services['service'].str.strip().str.title()
df_beds_services['month']   = df_beds_services['month'].str.strip().str.title()
# Validate bed numbers
df_beds_services['available_beds']    = df_beds_services['available_beds'].clip(lower=0)
df_beds_services['patients_admitted'] = df_beds_services['patients_admitted'].clip(lower=0)
df_beds_services['patients_refused']  = df_beds_services['patients_refused'].clip(lower=0)
# Calculate bed utilization rate
df_beds_services['bed_utilization_rate'] = (
    df_beds_services['patients_admitted'] / df_beds_services['available_beds'] * 100
).round(2)
print(f'Beds Services cleaned: {df_beds_services.shape}')
display(df_beds_services.head(3))

## Step 5: Clean Dataset 3 – Readmission Data

In [ ]:
# --- Clean Readmission Admission Data ---
df_readmission.columns = [c.strip().lower().replace(' ', '_').replace('.', '') for c in df_readmission.columns]
df_readmission = df_readmission.drop_duplicates(subset='mrd_no')

# Fix date columns
df_readmission['doa'] = pd.to_datetime(df_readmission['doa'], errors='coerce')  # Date of Admission
df_readmission['dod'] = pd.to_datetime(df_readmission['dod'], errors='coerce')  # Date of Discharge

# Rename for clarity
df_readmission = df_readmission.rename(columns={
    'mrd_no': 'patient_id',
    'doa': 'admission_date',
    'dod': 'discharge_date',
    'type_of_admission-emergency/opd': 'admission_type',
    'duration_of_stay': 'length_of_stay_days'
})

# Validate age
df_readmission = df_readmission[(df_readmission['age'] >= 0) & (df_readmission['age'] <= 120)]

# Standardize gender
df_readmission['gender'] = df_readmission['gender'].str.strip().str.upper()

# Validate LOS
df_readmission = df_readmission[df_readmission['length_of_stay_days'] >= 0]

# Readmission flag based on outcome
df_readmission['readmission_flag'] = df_readmission['outcome'].apply(
    lambda x: 1 if str(x).strip().upper() in ['DAMA', 'EXPIRY'] else 0
)

print(f'Readmission data cleaned: {df_readmission.shape}')
display(df_readmission[['patient_id','admission_date','discharge_date','age','gender','admission_type','length_of_stay_days','outcome','readmission_flag']].head(3))

## Step 6: Clean Dataset 4 – Healthcare Dataset

In [ ]:
# --- Clean Healthcare Dataset ---
df_healthcare.columns = [c.strip().lower().replace(' ', '_') for c in df_healthcare.columns]
df_healthcare = df_healthcare.drop_duplicates()

df_healthcare['date_of_admission'] = pd.to_datetime(df_healthcare['date_of_admission'], errors='coerce')
df_healthcare['discharge_date']    = pd.to_datetime(df_healthcare['discharge_date'], errors='coerce')
df_healthcare['length_of_stay_days'] = (df_healthcare['discharge_date'] - df_healthcare['date_of_admission']).dt.days

# Standardize text fields
df_healthcare['gender']           = df_healthcare['gender'].str.strip().str.title()
df_healthcare['admission_type']   = df_healthcare['admission_type'].str.strip().str.title()
df_healthcare['medical_condition']= df_healthcare['medical_condition'].str.strip().str.title()
df_healthcare['hospital']         = df_healthcare['hospital'].str.strip().str.title()
df_healthcare['doctor']           = df_healthcare['doctor'].str.strip().str.title()
df_healthcare['test_results']     = df_healthcare['test_results'].str.strip().str.title()

# Validate age
df_healthcare = df_healthcare[(df_healthcare['age'] >= 0) & (df_healthcare['age'] <= 120)]

# Validate LOS
df_healthcare = df_healthcare[df_healthcare['length_of_stay_days'] >= 0]

# Add year and month columns
df_healthcare['admission_year']  = df_healthcare['date_of_admission'].dt.year
df_healthcare['admission_month'] = df_healthcare['date_of_admission'].dt.month_name()

print(f'Healthcare Dataset cleaned: {df_healthcare.shape}')
display(df_healthcare.head(3))

## Step 7: Final Missing Value Check After Cleaning

In [ ]:
datasets = {
    'HMIS Patients':        df_patients,
    'HMIS BedRecords':      df_bed_records,
    'HMIS Department':      df_department,
    'Beds Patients':        df_beds_patients,
    'Beds Services':        df_beds_services,
    'Readmission Data':     df_readmission,
    'Healthcare Dataset':   df_healthcare
}

print('=== POST-CLEANING MISSING VALUE SUMMARY ===')
for name, df in datasets.items():
    total_cells    = df.shape[0] * df.shape[1]
    missing_cells  = df.isnull().sum().sum()
    missing_pct    = round(missing_cells / total_cells * 100, 2)
    status = '✅' if missing_pct < 2 else '⚠️'
    print(f'{status} {name}: {missing_pct}% missing ({missing_cells}/{total_cells} cells)')

## ✅ Data Cleaning Complete!
All datasets are cleaned. Proceed to **03_data_normalization.ipynb**